# Bolyai Labs – Feladatkártya #2: A hívásnapló megtisztítása

> Adatelemzőként szeretném, hogy a hívásnapló tiszta és egységes legyen, hogy a következő blokkokban a modellek megbízható adatból tanuljanak.

**Így dolgozz:** futtasd a cellákat sorban (Shift+Enter). Ahol `...  # TODO` áll, ott a `...` helyére írd a megoldást. A végén az **önellenőrző cella** megmondja, jó-e az eredmény.

**Mentés:** File → Save a copy in GitHub → a saját repód, `main` ág, a fájlnév maradjon `adattisztitas.ipynb`. Commitüzenet: `feat: hívásnapló tisztítása (feladatkártya #2)`.

In [1]:
from pathlib import Path

import pandas as pd

NYERS = Path("data/hivasok_nyers.csv")
SABLON_URL = "https://raw.githubusercontent.com/Kizsi2025/bolyai-labs-02-adat/main/data/hivasok_nyers.csv"

# Helyben (és a CI-ban) a repó fájlját olvassuk, Colabban a sablonrepóból töltjük le.
df = pd.read_csv(NYERS if NYERS.exists() else SABLON_URL, dtype=str)
print(df.shape)
df.head(10)

(232, 6)


,hivas_id,datum,modell,valaszido_ms,tokenek,sikeres
0,94,2026-10-12,nagyobb,911,980,no
1,4,2026-10-01,nagyobb,981,740,True
2,181,2026-10-23,nagyobb,914,787,1
3,130,2026-10-17,kisebb,355,459,yes
4,60,2026-10-08,kisebb,435,420,igen
5,87,2026-10-11,nagyobb,1074,1056,1
6,7,2026-10-01,nagyobb,1219,449,igen
7,177,2026-10-23,kisebb,hiba,432,1
8,188,2026-10-24,kisebb,348,382,1
9,198,2026-10-25,nagyobb,941,875,yes


## 1. Feltárás – mit rontottak el az adatban?

Futtasd az alábbi cellát, és nézd meg alaposan a kimenetet. Utána válaszolj a kérdésekre a következő szöveges cellában.

In [2]:
print("Hiányzó értékek oszloponként:")
print(df.isna().sum(), end="\n\n")
print("Teljesen egyező (duplikált) sorok száma:", df.duplicated().sum(), end="\n\n")
print("A modell oszlop írásmódjai:")
print(df["modell"].value_counts(), end="\n\n")
print("A sikeres oszlop írásmódjai:")
print(df["sikeres"].value_counts(), end="\n\n")
print("Nem szám válaszidők:")
print(df.loc[pd.to_numeric(df["valaszido_ms"], errors="coerce").isna() & df["valaszido_ms"].notna(), "valaszido_ms"].value_counts())

Hiányzó értékek oszloponként:
hivas_id         0
datum            0
modell           0
valaszido_ms    13
tokenek          0
sikeres          0
dtype: int64

Teljesen egyező (duplikált) sorok száma: 12

A modell oszlop írásmódjai:
modell
nagyobb     111
kisebb       90
nagyobb       7
 NAGYOBB      6
kisebb        5
Nagyobb       5
 Kisebb       4
KISEBB        4
Name: count, dtype: int64

A sikeres oszlop írásmódjai:
sikeres
1        104
igen      43
yes       35
True      33
0          7
no         4
nem        4
False      2
Name: count, dtype: int64

Nem szám válaszidők:
valaszido_ms
hiba    3
Name: count, dtype: int64


**Válaszolj ide (dupla kattintás a cellára):**

1. Hány duplikált sor van az adatban?
2. Hányféleképpen írták a modell nevét?
3. Milyen nem szám értékek vannak a válaszidő oszlopban?
4. Szerinted reális-e a 999 999 ms-os válaszidő? Miért?

## 2. Tisztítás – hét lépés, ebben a sorrendben

A sorrend számít: ha felcseréled, más eredményt kapsz, és az önellenőrzés jelezni fogja.

In [9]:
# a) Duplikált sorok törlése
df = pd.read_csv("./data/hivasok_nyers.csv").drop_duplicates()  # TODO: a teljesen egyező sorokból csak egy maradjon (drop_duplicates)
print(df.shape)

(220, 6)


In [ ]:
# b) A modellnév egységesítése: szóközök le, minden kisbetűs
df["modell"] = ... # TODO: .str.strip() és .str.lower()
df["modell"].value_counts()

In [ ]:
# c) A sikeres oszlop 0/1 számmá alakítása
ERTEKEK = {"1": 1, "igen": 1, "yes": 1, "true": 1, "0": 0, "nem": 0, "no": 0, "false": 0}
df["sikeres"] = ...  # TODO: kisbetűsítsd (.str.lower()), majd alakítsd át a szótárral (.map(ERTEKEK))
df["sikeres"].value_counts()

In [ ]:
# d) A válaszidő számmá alakítása – ami nem szám (n/a, hiba, üres), abból hiányzó érték (NaN) lesz
df["valaszido_ms"] = ...  # TODO: pd.to_numeric(..., errors="coerce")
df["tokenek"] = pd.to_numeric(df["tokenek"])
df["valaszido_ms"].isna().sum()

In [ ]:
# e) Kiugró értékek eltávolítása: 50 ms alatt és 10 000 ms fölött mérési hiba
ALSO, FELSO = 50, 10_000
df = ...  # TODO: maradjanak a sorok, ahol a válaszidő hiányzik VAGY ALSO és FELSO között van (.isna(), .between())
print(df.shape)

In [ ]:
# f) Hiányzó válaszidők pótlása a saját modell mediánjával
df["valaszido_ms"] = ...  # TODO: .fillna(df.groupby("modell")["valaszido_ms"].transform("median"))
df["valaszido_ms"].isna().sum()

In [ ]:
# g) Min-max normalizálás: a legkisebb válaszidő 0, a legnagyobb 1 legyen
v = df["valaszido_ms"]
df["valaszido_norm"] = ...  # TODO: (v - v.min()) / (v.max() - v.min())
df["valaszido_norm"].describe()

## 3. Önellenőrzés

Ha ez a cella hiba nélkül lefut, a tisztítás jó. Ha hibát ír, az üzenet megmondja, melyik lépést nézd át.

In [ ]:
assert not df.duplicated().any(), "a) Maradtak duplikált sorok."
assert set(df["modell"]) == {"kisebb", "nagyobb"}, f"b) A modellnevek még nem egységesek: {sorted(set(df['modell']))}"
assert set(df["sikeres"]) <= {0, 1}, "c) A sikeres oszlopban nem csak 0 és 1 van (hiányzik egy írásmód a szótárból?)."
assert df["valaszido_ms"].between(50, 10_000).all(), "e) Maradt kiugró érték, vagy f) maradt hiányzó válaszidő."
assert abs(df["valaszido_norm"].min()) < 1e-9 and abs(df["valaszido_norm"].max() - 1) < 1e-9, "g) A normalizált érték nem 0 és 1 közé esik."
print(f"Minden ellenőrzés rendben! {len(df)} tiszta sor.")

## 4. Mentés

Ez a cella elmenti a tiszta adatot. A CI ugyanezt a notebookot futtatja le, és a mentett fájlt ellenőrzi.

In [ ]:
Path("data").mkdir(exist_ok=True)
df.to_csv("data/hivasok_tiszta.csv", index=False)
print("Mentve: data/hivasok_tiszta.csv")